<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-Core-ISMMS/ImageAnalysisCourse/blob/2026-workshop/notebooks/00_setup_self_check.ipynb)

*Click the badge to open this notebook in Google Colab. For best performance, switch to a GPU runtime: Runtime → Change runtime type → T4 GPU.*

# Notebook 00 — Setup and Self-Check

**Purpose.** Verify that your Colab (or local Jupyter) environment is ready for the workshop, and exercise the basic Python and image-data skills the labs assume. Run this notebook before the workshop day. Re-run on the morning of the workshop to confirm nothing has drifted.

**Estimated time.** 20–30 minutes.

By the end of this notebook you should know whether you are ready for the workshop or whether to attend the optional ramp-up evening session.

## Step 1 — Detect your environment

In [ ]:
import sys
import platform

IN_COLAB = "google.colab" in sys.modules
print("Python    :", sys.version.split()[0])
print("Platform  :", platform.platform())
print("Runtime   :", "Google Colab" if IN_COLAB else "Local Jupyter")

## Step 2 — Install the dependencies

This cell installs only what the self-check needs (numpy, matplotlib, scikit-image, requests). The lab notebooks each install their own additional dependencies.

In [ ]:
# Idempotent install. If you already have these, pip will skip them.
%pip install --quiet numpy matplotlib scikit-image requests tifffile
print("Dependencies installed.")

## Step 3 — Download a canonical sample image

We'll use one of the Cellpose example images. If your network blocks the download, the next cell falls back to a synthetic image so the rest of the notebook still runs.

In [ ]:
import os
import requests
import numpy as np

SAMPLE_URL = "http://www.cellpose.org/static/data/img02.png"
SAMPLE_PATH = "sample.png"

try:
    r = requests.get(SAMPLE_URL, timeout=20)
    r.raise_for_status()
    with open(SAMPLE_PATH, "wb") as f:
        f.write(r.content)
    print(f"Downloaded {SAMPLE_PATH} ({len(r.content)} bytes)")
    download_ok = True
except Exception as e:
    print(f"Download failed: {e}")
    print("Falling back to a synthetic image so the notebook still runs.")
    download_ok = False

## Step 4 — Load and inspect the image

In [ ]:
from skimage import io as skio

if download_ok:
    img = skio.imread(SAMPLE_PATH)
else:
    # Synthetic fallback: a small image with a few bright blobs
    rng = np.random.default_rng(42)
    img = np.zeros((200, 200), dtype=np.uint8)
    for _ in range(8):
        cy, cx = rng.integers(20, 180, size=2)
        r = rng.integers(8, 16)
        Y, X = np.ogrid[:200, :200]
        mask = (Y - cy) ** 2 + (X - cx) ** 2 <= r ** 2
        img[mask] = 255
    img = img + rng.normal(0, 10, img.shape).astype(np.int16)
    img = np.clip(img, 0, 255).astype(np.uint8)

print("shape :", img.shape)
print("dtype :", img.dtype)
print("min   :", int(img.min()))
print("max   :", int(img.max()))
print("mean  :", float(img.mean().round(2)))

## Step 5 — Predict before you display

Before you run the next cell, look at the shape, dtype, and stats above and form a mental picture. What do you expect to see?

Then run the cell.

In [ ]:
import matplotlib.pyplot as plt

# Pick a sensible display contrast
p1, p99 = np.percentile(img, [1, 99])

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(img, cmap="gray", vmin=p1, vmax=p99)
ax.set_title(f"Sample image  ({img.shape[0]}x{img.shape[1]})")
ax.axis("off")
plt.show()

## Step 6 — Numpy slicing exercise

A short hands-on check. The cell below extracts a subregion and computes its mean, then compares to the whole-image mean. Skim the code, predict whether the subregion mean will be higher or lower, then run.

In [ ]:
# Take a 60x60 subregion from the center of the image
h, w = img.shape[:2]
cy, cx = h // 2, w // 2
sub = img[cy - 30:cy + 30, cx - 30:cx + 30]

sub_mean = sub.mean()
img_mean = img.mean()
print(f"Whole-image mean : {img_mean:.2f}")
print(f"Subregion mean   : {sub_mean:.2f}")
print(f"Difference       : {sub_mean - img_mean:+.2f}")

## Step 7 — Self-grading and recommendation

This cell summarises the self-check and tells you whether to attend the optional ramp-up evening session.

In [ ]:
env_ok = True
load_ok = "img" in dir() and img.size > 0
slice_ok = "sub" in dir() and sub.size > 0

print("=" * 50)
print("Self-check summary")
print("=" * 50)
print(f"  Environment ready : {'OK' if env_ok else 'MISSING'}")
print(f"  Image loaded      : {'OK' if load_ok else 'MISSING'}")
print(f"  Slicing exercise  : {'OK' if slice_ok else 'MISSING'}")
print()

if env_ok and load_ok and slice_ok:
    print("Recommendation: you are ready for the workshop.")
elif env_ok and load_ok:
    print("Recommendation: you are likely ready. Consider the ramp-up if you")
    print("found the slicing exercise unfamiliar.")
else:
    print("Recommendation: please attend the ramp-up evening session.")
    print("Email the instructors if you get stuck before the workshop.")

## What this notebook did *not* do

- Train any model
- Perform any segmentation
- Touch any GPU
- Use any of the workshop's lab tools (Cellpose, Noise2Void, segment-anything)

Those are deliberately reserved for the workshop day. The self-check is just a runtime sanity check.

If you want to push further before the workshop, look at the optional cells in the ramp-up evening notebooks (`ramp-up/notebooks/`).